# 13. Asking a language model directly (zero-shot, GPU)

The baseline for "just ask the model": Mistral-Small-24B **Instruct** (the instruct version of the frozen model of step
7, 4 bit) reads each anonymous article with a description of NewsGuard's nine criteria and answers with the score
(0–100) of the outlet that published it. Nothing is trained; greedy decoding; the outlet score is the mean over its
articles, as for every model (prompt and parsing in `zeroshot.py`). Outputs in `predictions/regression/LLM-zero-shot/`,
so steps 8, 9 and 15 include it like the models of step 7.

In [ ]:
LIMIT = 0                     # > 0: only the first LIMIT articles, to test the prompt (nothing is saved)
GPU_GB = 20

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO
import zeroshot as ZS

art, outlets = MO.load()
print(ZS.SYSTEM, "\n\n" + ZS.USER.format(text="<article>"))

In [ ]:
MO.wait_for_gpu(GPU_GB, LOG)
res = ZS.run(art, LOG, limit=LIMIT or None)
print(f"{len(res):,} answers; without an integer: {res.prediction.isna().sum()}")
res.answer.value_counts().head(15)

In [ ]:
res.prediction.describe().round(1).to_frame("predicted score per article").T